# HW4: Reading the Verdicts

**COMPSS 211A | Fall 2026 | Student copy**

In this homework you use TF-IDF to ask a real question about real text: **what kinds of situations does an online community judge harshly?** You'll work with about 3,800 posts from Reddit's r/AmItheAsshole. Along the way you'll choose which posts to keep, find problems in the words TF-IDF produces, and change the code to deal with them.

**Due:** Sunday, November 8, 2026 at 11:59 p.m.

Some of the code will look familiar from Week 8 and Lab 8. This time you won't just run it: you'll predict what it does, add to it, and change it. When a cell asks for a prediction, write your guess **before** you run anything. Guessing wrong is fine; that's often where you learn the most.

Keep the variable names we ask for, because we look for them when grading.

## The data

[r/AmItheAsshole](https://www.reddit.com/r/AmItheAsshole/) ("AITA") is a Reddit forum where people describe a conflict from their own life and ask strangers whether they were in the wrong. Commenters reply with a verdict, and after about a day the post gets a label (a *flair*) based on the top-voted comment:

| Flair | Short form | Meaning |
| --- | --- | --- |
| Not the A-hole | NTA | The poster was not in the wrong. |
| Asshole | YTA | "You're the asshole": the poster was in the wrong. |
| Everyone Sucks | ESH | Everyone involved behaved badly. |
| No A-holes here | NAH | Nobody behaved badly; it's a real disagreement. |

The file `data/aita_top_subs.csv` has 5,000 real posts from 2018 to 2021. They are among the **most upvoted** posts on the forum, so they are not a random sample of AITA, let alone of Reddit or the public. The file has no usernames. Even so, these are real people's stories: don't try to find out who wrote them, and don't quote them outside this class.

## What you will practice

- Deciding which rows belong in your analysis, and saying what you left out.
- Predicting what cleaning code does, then checking.
- Building TF-IDF features and knowing what the rows and columns of the matrix are.
- Spotting junk in your top words and changing the settings to fix it.
- Comparing two groups of documents and reading the posts behind a word.

## Helpful references

- **Week 8 reference** (`lessons/week08_nlp-fundamentals/nlp_features_reference.md`) explains `TfidfVectorizer` and its settings. Week 8 used AITA *comments*; here you use the *posts*.
- **Lab 8** has the first versions of `clean_comment` and `rank_terms`.
- **What TF-IDF means.** A word gets a high TF-IDF score in a post when it appears in that post but is rare in the other posts. A high score means the word is *distinctive*, not that it is *important*.
- **The TF-IDF matrix.** Each **row** is one post, in the same order as your table. Each **column** is one word. So picking rows of the matrix is the same as picking posts.

## AI and collaboration policy

Don't use AI to write your code or your memo. You may use it to explain an error message if you say so in a note in your notebook. Talking through ideas with classmates is fine; share ideas, not code.

In [ ]:
from pathlib import Path
import json
import os
import sys

import re
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

SUPPORTED_PYTHON = (3, 13)
if sys.version_info[:2] != SUPPORTED_PYTHON:
    print(
        "Setup note: this course is tested with Python 3.13; "
        f"you are running {sys.version.split()[0]}."
    )

def find_course_root():
    """Find the cloned repository when this notebook is running locally."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "pyproject.toml").exists() and (folder / "data").is_dir():
            return folder
    return None

LOCAL_COURSE_ROOT = find_course_root()
COURSE_ROOT = LOCAL_COURSE_ROOT or Path.cwd()
DATA_DIR = (
    LOCAL_COURSE_ROOT / "data"
    if LOCAL_COURSE_ROOT
    else COURSE_ROOT / "compss211_data"
)
GENERATED_DIR = COURSE_ROOT / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "macss-berkeley/compss-211a/main/data"
)

def course_data_path(filename):
    """Use a local course file, or download it when running in Colab."""
    path = DATA_DIR / filename
    if not path.exists():
        from urllib.request import urlretrieve

        urlretrieve(f"{DATA_BASE_URL}/{filename}", path)
    return path

print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

## 1. Decide which posts to keep

Not every row is usable. Some posts were removed by moderators or deleted by their authors, so the text just says `[removed]` or `[deleted]`. Some flairs aren't verdicts at all (`UPDATE`, `META`), and some posts never got a flair. Run the cell below to see what you're dealing with.

In [ ]:
VERDICT_LABELS = {
    "Not the A-hole": "NTA",
    "not the a-hole": "NTA",
    "Asshole": "YTA",
    "Everyone Sucks": "ESH",
    "No A-holes here": "NAH",
}

raw_posts = pd.read_csv(course_data_path("aita_top_subs.csv"))
print(len(raw_posts), "posts in the file")
print(raw_posts["selftext"].isin(["[removed]", "[deleted]"]).sum(), "posts removed or deleted")

raw_posts["flair_text"].value_counts(dropna=False)

**Your turn:** make a table called `posts` that keeps only the rows where:

- `flair_text` is one of the keys in `VERDICT_LABELS` (hint: `.isin(...)`), and
- `selftext` is not missing and is not `[removed]` or `[deleted]` (hint: `~` flips True and False).

Then add two columns:

- `verdict`: the short form (NTA, YTA, ESH, NAH). Hint: `.map(VERDICT_LABELS)`.
- `text`: the title and the post body joined with a space, since the title often says the most (`posts["title"] + " " + posts["selftext"]`).

Use `.copy()` when you filter, so you're working on your own table. Finally, show how many posts you kept and what share has each verdict (`value_counts(normalize=True)`).

In [ ]:
# Your turn
posts = ...

## 2. Predict what the cleaning does

Here is `clean_comment` from Lab 8, renamed `clean_post`. Read it line by line.

`re.sub(pattern, replacement, text)` finds every piece of `text` that matches `pattern` and replaces it. In the pattern `[^a-z0-9\s'-]`, the square brackets mean "any one character from this set", and the `^` at the start flips it to "any character **not** in this set". So the first `re.sub` replaces every character that is *not* a lowercase letter a to z, a digit, a space, an apostrophe, or a hyphen.

In [ ]:
def clean_post(text):
    """Lowercase the text, remove most punctuation, and tidy up spaces."""
    lowered = str(text).lower()
    # Replace anything that isn't a-z, 0-9, a space, an apostrophe, or a hyphen with a space.
    no_punctuation = re.sub(r"[^a-z0-9\s'-]", " ", lowered)
    # Turn runs of spaces into a single space, and trim both ends.
    return re.sub(r"\s+", " ", no_punctuation).strip()

**Your turn:** *without running `clean_post`*, write down what you think it returns for each string below. Type your guess as the value in the dictionary. Then run the cell: it checks each guess for you.

Worked example: `"AITA for skipping my sister's wedding?"` becomes `"aita for skipping my sister's wedding"`.

In [ ]:
my_predictions = {
    "WIBTA if I told my MIL to leave?": "...",
    "My fiancé (27F) said NO.": "...",
    "EDIT: ok, I'm the a**hole": "...",
}

for original, guess in my_predictions.items():
    actual = clean_post(original)
    result = "correct" if guess == actual else "different"
    print(f"{result:9} | {original!r} -> {actual!r}")

In [ ]:
# Now clean every post. .apply() runs clean_post once for each row.
posts["clean_text"] = posts["text"].apply(clean_post)
posts[["text", "clean_text"]].head()

### Your response


Answer in two or three sentences:

1. Which prediction did you get wrong (if any), and what did you misunderstand about the code?
2. What happened to `fiancé`? Explain *which part* of the code caused it. How many posts might this affect? (Try `posts["text"].str.contains("é").sum()`.)


> Write your response here, then delete this line.

## 3. Extend `rank_terms`, then build TF-IDF

The Lab 8 version of `rank_terms` gives you each word's average TF-IDF score across all posts. But a word can have a high average for two very different reasons: it appears in *many* posts, or it scores very high in just a *few*. To tell which, you need to know how many posts each word appears in.

**Your turn:** add a column `n_posts` to `rank_terms` that counts, for each word, how many posts contain it.

Hints:
- `matrix > 0` gives a True/False matrix: True where a post contains a word.
- Summing a True/False matrix down the rows, `.sum(axis=0)`, counts the Trues in each column. Remember: columns are words.
- Wrap the result in `np.asarray(...).ravel()`, just like the line that computes `average_scores`, to turn it into a plain list of numbers.

In [ ]:
def rank_terms(vectorizer, matrix, n=25):
    """Return the n words with the highest average TF-IDF score."""
    # Average score of each word (column) across all posts (rows).
    average_scores = np.asarray(matrix.mean(axis=0)).ravel()

    # Your turn: count how many posts (rows) contain each word (column).
    posts_with_word = ...

    ranked = pd.DataFrame({
        "term": vectorizer.get_feature_names_out(),
        "mean_tfidf": average_scores,
        # Your turn: add the n_posts column here.
    })
    return ranked.sort_values("mean_tfidf", ascending=False).head(n).reset_index(drop=True)

Now build the features. Use `TfidfVectorizer(stop_words="english", min_df=5)`. `stop_words="english"` drops very common words like "the" and "is". `min_df=5` drops words that appear in fewer than five posts (typos, names, and other rare words). `fit_transform` learns the vocabulary *and* builds the matrix in one step.

**Predict first:** write down three words you expect near the top of the list for AITA posts. Then build `tfidf_vectorizer` and `tfidf_matrix` from `posts["clean_text"]`, print the matrix shape, and make `top_terms` with your `rank_terms`.

In [ ]:
# My three predicted top words:
#

# Your turn
tfidf_vectorizer = ...
tfidf_matrix = ...
top_terms = ...

## 4. Clean up the vocabulary

Look closely at `top_terms`. Some of the top "words" aren't telling you much about the stories:

- **Forum labels** like `aita`. Nearly every title starts with "AITA for...", so it tells you nothing about the situation.
- **Word pieces** like `don`, `didn`, and `ve`. `clean_post` keeps the apostrophe in "don't", but `TfidfVectorizer` has its own rule for splitting text into words, and that rule treats the apostrophe as a break. So "don't" becomes `don` plus a `t`, and the one-letter `t` is thrown away.
- Anything else that looks odd to you, like `fianc` from Task 2.

You can add your own words to the stop-word list. Week 8 did this to *keep* "not"; here you'll use it to *remove* words:

```python
custom_stop_words = list(ENGLISH_STOP_WORDS | {"aita", "wibta"})
TfidfVectorizer(stop_words=custom_stop_words, min_df=5)
```

The `|` joins two sets together.

**Your turn:**

1. Make `custom_stop_words` with the words you decide to remove. Look beyond the top 25 too (try `rank_terms(..., n=60)`). You don't have to remove every word-piece: `didn` and `don` are what's left of *didn't* and *don't*, so removing them removes negation from your features. Your call; you'll explain it in the memo.
2. **Predict first:** if you add *k* new words, will the number of columns drop by exactly *k*? Write your guess as a comment.
3. Build `custom_vectorizer`, `custom_matrix`, and `top_terms_custom`, print the shape, and run the comparison.

In [ ]:
# My prediction (does the column count drop by exactly k?):
#

# Your turn
custom_stop_words = ...
custom_vectorizer = ...
custom_matrix = ...
top_terms_custom = ...

term_comparison = pd.DataFrame({
    "before": top_terms["term"],
    "after": top_terms_custom["term"],
})
term_comparison

## 5. Compare YTA and NTA posts

Now the actual question: do posts judged **YTA** talk about different things than posts judged **NTA**?

The trick: since each row of the matrix is one post, you can pick out the rows for one verdict with a True/False mask and pass just those rows to `rank_terms`:

```python
is_yta = (posts["verdict"] == "YTA").to_numpy()
rank_terms(custom_vectorizer, custom_matrix[is_yta], n=15)
```

The table, the vectorizer, and the matrix must all come from the same version, so the rows line up.

**Your turn:** make `top_yta` and `top_nta` this way and put their `term` columns side by side. How different are the two lists?

In [ ]:
# Your turn
is_yta = ...
is_nta = ...
top_yta = ...
top_nta = ...

pd.DataFrame({"YTA": top_yta["term"], "NTA": top_nta["term"]})

The two lists probably look almost the same. Words like `told` and `said` are common in *every* kind of post, so they top both lists. To see what sets the groups apart, compare the **difference** in average scores: YTA average minus NTA average. The helper below does that. Words at the top lean YTA; words at the bottom lean NTA.

In [ ]:
def compare_groups(vectorizer, matrix, mask_a, mask_b, n=15):
    """Return the words whose average score differs most between two groups of rows."""
    average_a = np.asarray(matrix[mask_a].mean(axis=0)).ravel()
    average_b = np.asarray(matrix[mask_b].mean(axis=0)).ravel()
    difference = pd.DataFrame({
        "term": vectorizer.get_feature_names_out(),
        "difference": average_a - average_b,
    }).sort_values("difference", ascending=False)
    return pd.DataFrame({
        "leans_a": difference["term"].head(n).to_numpy(),
        "leans_b": difference["term"].tail(n)[::-1].to_numpy(),
    })

**Your turn:**

1. Call `compare_groups` for YTA (group a) versus NTA (group b) and save the result as `verdict_terms`.
2. Pick **one** word from either column. Show the `title` of five posts with that verdict that contain the word (hint: `.str.contains(...)` on `clean_text`, combined with a verdict condition). Read a couple of the full posts too. What does the word tell you about the kinds of situations in that group?

One more thing to check: if a word in the list seems to *give away* the verdict, find out where it appears in the posts. Many posters add an "EDIT:" after they've seen the comments. What would that mean for your comparison?

In [ ]:
# Your turn
verdict_terms = ...

## 6. Write a short memo

Write five to seven sentences. Include:

1. **Which posts you kept.** How many, and what did you leave out? Could leaving out removed or deleted posts change your results?
2. **One cleaning decision.** Pick one (`fianc`, `don`/`didn`, `aita`, or another word you removed or kept) and explain your choice.
3. **What differs.** Name one word that leans YTA or NTA and describe what it meant in the posts you read.
4. **The limits.** What can't you conclude? Think about who these posts and verdicts come from, and whether a word being associated with a verdict means it *causes* the verdict.

### Your response

**Question:** What distinguishes posts judged YTA from posts judged NTA, and how much should anyone trust that finding?

> Write your response here, then delete this line.